# Figure 4 — submission notebook

Authoritative panels B–F are independently replotted from local HDF5 data before compilation.

## Plot-only execution contract

This notebook renders existing intermediate numerical results only. It does not recompute upstream analysis.


In [ ]:
PLOT_ONLY_FIGURE = "Figure_4"
ANALYSIS_RECOMPUTED = False
ALLOWED_INPUT_SUFFIXES = {".h5ad", ".h5", ".csv", ".gz", ".parquet", ".json"}
print(PLOT_ONLY_FIGURE, "plot-only mode; analysis_recomputed=", ANALYSIS_RECOMPUTED)

In [ ]:
# Conventional Jupyter inline plotting: live Matplotlib figures are rendered by plt.show().
get_ipython().run_line_magic('matplotlib', 'inline')

# Figure 4 — in-house TNBC cell identity, H-cluster markers, CNV, and TNBCtype

Strictly plot-only notebook. Every panel reads the final local Figure 4 HDF5 package; no upstream analysis data are accessed at plotting time.


## Setup, validation, and shared style

In [ ]:
from pathlib import Path
import platform, warnings

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, TwoSlopeNorm
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd

# ─── Figure parameters (user-editable) ───────────────────────────────
FONT_FAMILY = ["Arial", "Helvetica", "DejaVu Sans"]
PANEL_LABEL_SIZE_PT = 10
AXIS_TITLE_SIZE_PT = 9
TICK_LABEL_SIZE_PT = 7
LEGEND_SIZE_PT = 7
COLORBAR_LABEL_SIZE_PT = 8
UMAP_POINT_SIZE = 0.30
UMAP_ALPHA = 0.80
RASTERIZED = True
H_HEATMAP_CMAP = "bwr"
H_HEATMAP_VMIN = -3
H_HEATMAP_VMAX = 3
PANEL_A_FIGSIZE = (7.1, 5.8)
PANEL_B_FIGSIZE = (16.0, 6.8)
PANEL_C_FIGSIZE = (14.0, 10.0)
METADATA_TRACK_HEIGHT_RATIO = 0.030
CLUSTER_BAR_HEIGHT_RATIO = 0.055
MARKER_SOURCE_WIDTH_RATIO = 0.008
COLORBAR_WIDTH_RATIO = 0.022
LEGEND_WIDTH_RATIO = 0.18
ANNOTATION_TRACK_LABEL_SIZE_PT = 5.5
ANNOTATION_LEGEND_SIZE_PT = 5.5
ANNOTATION_HSPACE = 0.002
ANNOTATION_WSPACE = 0.018
MARKER_TITLE_SIZE_PT = 8
MARKER_COLORBAR_TICK_SIZE_PT = 6
MARKER_COLORBAR_SHRINK = 0.68
MARKER_COLORBAR_PAD = 0.015
UMAP_LIMIT_PADDING = 0.02
PANEL_D_FIGSIZE = (11.0, 6.2)
EXPECTED_CELLS = 69_391
# ─────────────────────────────────────────────────────────────────────

NOTEBOOK_DIR = Path.cwd().resolve()
PLOT_DATA = NOTEBOOK_DIR / "data/fig4_plot_data.h5"
PATIENT_MAPPING_DATA = NOTEBOOK_DIR / "data/patient_id_mapping.csv"
OUTPUT_DIR = NOTEBOOK_DIR / "figs/panel/Figure_4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DPI = 300


if not PLOT_DATA.is_file():
    raise FileNotFoundError(f"Missing Figure 4 plot-only bundle: {PLOT_DATA}")
if not PATIENT_MAPPING_DATA.is_file():
    raise FileNotFoundError(f"Missing local Figure 4 patient mapping: {PATIENT_MAPPING_DATA}")
patient_mapping = pd.read_csv(PATIENT_MAPPING_DATA, dtype=str)
if list(patient_mapping.columns) != ["display_id", "patient_id"] or patient_mapping["display_id"].tolist() != [f"P{i}" for i in range(1, 8)]:
    raise RuntimeError("Invalid local Figure 4 patient display mapping")
PATIENT_DISPLAY = patient_mapping.set_index("patient_id")["display_id"].to_dict()

def read_strings(dataset):
    return np.asarray(dataset.asstr()[:], dtype=str)

required_groups = {"shared", "panel_a", "panel_b", "panel_c", "panel_d", "panel_e", "panel_f", "panel_g"}
with h5py.File(PLOT_DATA, "r") as bundle:
    missing_groups = sorted(required_groups.difference(bundle.keys()))
    if missing_groups:
        raise RuntimeError(f"Plot-data bundle lacks groups: {missing_groups}")
    cell_ids = pd.Index(read_strings(bundle["shared/cell_id"]), name="cell_id")
    umap = np.asarray(bundle["shared/umap"][:], dtype=np.float32)
    schema_version = str(bundle.attrs.get("schema_version", "unknown"))
if len(cell_ids) != EXPECTED_CELLS or umap.shape != (EXPECTED_CELLS, 2) or cell_ids.has_duplicates:
    raise RuntimeError("Invalid shared cell IDs or UMAP coordinates in plot-data bundle")

mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": FONT_FAMILY,
    "font.size": TICK_LABEL_SIZE_PT, "axes.titlesize": PANEL_LABEL_SIZE_PT,
    "axes.labelsize": AXIS_TITLE_SIZE_PT, "xtick.labelsize": TICK_LABEL_SIZE_PT,
    "ytick.labelsize": TICK_LABEL_SIZE_PT, "legend.fontsize": LEGEND_SIZE_PT,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "figure.facecolor": "white",
})
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Validated plot-only bundle schema {schema_version} | {len(cell_ids):,} cells | Python {platform.python_version()} | matplotlib {mpl.__version__}")
print("Rendering path: HDF5 + NumPy/Pandas + Matplotlib only; no analysis or preprocessing is run.")

## Supplementary Figure 4 — authoritative T/R/S/Q/N/P/M panels

All panels below are regenerated from staged intermediate numerical data.

In [ ]:
# Figure S4 plotting setup — staged numerical data only; no analysis is recomputed.
import json
import anndata as ad
import scanpy as sc
from matplotlib.lines import Line2D

S4_DATA = NOTEBOOK_DIR / "data/fig4_supplementary_plot_data.h5"
S4_OUT = NOTEBOOK_DIR / "outputs/individual"
S4_COMPILED = NOTEBOOK_DIR / "outputs/Figure_S4"
S4_MANIFEST = NOTEBOOK_DIR / "outputs/Figure_S4_manifest.json"
S4_OUT.mkdir(parents=True, exist_ok=True)

# ─── Figure S4 parameters (user-editable) ───────────────────────────
S4_WIDTH_IN = 11.0
S4_HEIGHT_IN = 16.0
S4_PANEL_TITLE_PT = 10
S4_AXIS_TITLE_PT = 8
S4_TICK_PT = 6.5
S4_LEGEND_PT = 6.2
S4_GENE_PT = 5.8
S4_UMAP_POINT_SIZE = 0.12
S4_UMAP_ALPHA = 0.72
S4_DOT_MIN_SIZE = 3.0
S4_DOT_MAX_SIZE = 48.0
S4_DPI = 300
S4_RASTERIZE_UMAPS = True
S4_EDITABLE_SVG_TEXT = False
# ─────────────────────────────────────────────────────────────────────

mpl.rcParams.update({"svg.fonttype": "none" if S4_EDITABLE_SVG_TEXT else "path"})

S4_CT_ORDER = [
    "Epithelial", "Epithelial_Luminal_Progenitor", "Epithelial_Luminal_Mature",
    "Epithelial_cycling", "Basal_Myoepithelial", "CAFs", "Macrophages",
    "T_cells", "B_cells", "Plasma_cells", "Endothelial", "pDC",
]
S4_CT_LABEL = {
    "Epithelial": "Epithelial", "Epithelial_Luminal_Progenitor": "Luminal Progenitor",
    "Epithelial_Luminal_Mature": "Luminal Mature", "Epithelial_cycling": "Cycling",
    "Basal_Myoepithelial": "Basal/Myoepithelial", "CAFs": "CAFs",
    "Macrophages": "Macrophages", "T_cells": "T cells", "B_cells": "B cells",
    "Plasma_cells": "Plasma cells", "Endothelial": "Endothelial", "pDC": "pDC",
}
S4_CT_COLORS = {
    "Epithelial": "#1f77b4", "Epithelial_Luminal_Progenitor": "#ff7f0e",
    "Epithelial_Luminal_Mature": "#2ca02c", "Epithelial_cycling": "#d62728",
    "Basal_Myoepithelial": "#9467bd", "CAFs": "#8c564b",
    "Macrophages": "#e377c2", "T_cells": "#7f7f7f", "B_cells": "#bcbd22",
    "Plasma_cells": "#17becf", "Endothelial": "#aec7e8", "pDC": "#ffbb78",
}
S4_CNV_CLUSTER_COLORS = [
    "#023fa5", "#7d87b9", "#bec1d4", "#d6bcc0", "#bb7784", "#8e063b",
    "#4a6fe3", "#8595e1", "#b5bbe3", "#e6afb9", "#e07b91", "#d33f6a",
    "#11c638", "#8dd593", "#c6dec7", "#ead3c6", "#f0b98d", "#ef9708",
    "#0fcfc0", "#9cded6", "#d5eae7", "#f3e1eb", "#f6c4e1", "#f79cd4",
    "#7f7f7f", "#c7c7c7",
]
S4_CNV_STATUS_ORDER = ["provisional_normal_CNV_low", "provisional_tumor_CNV_high"]
S4_CNV_STATUS_LABEL = {
    "provisional_normal_CNV_low": "CNV-low (provisional)",
    "provisional_tumor_CNV_high": "CNV-high shortlist",
}
S4_CNV_STATUS_COLORS = {
    "provisional_normal_CNV_low": "#bdbdbd",
    "provisional_tumor_CNV_high": "#7b3294",
}

def s4_strings(values):
    return np.asarray([value.decode() if isinstance(value, bytes) else str(value) for value in values])

if not S4_DATA.is_file():
    raise FileNotFoundError(S4_DATA)

with h5py.File(S4_DATA, "r") as bundle:
    s4_atlas_xy = np.asarray(bundle["atlas/umap"][:], dtype=np.float32)
    s4_atlas_celltype = s4_strings(bundle["atlas/celltype"][:])
    s4_composition_counts = np.asarray(bundle["atlas/composition_counts"][:], dtype=np.int64)
    s4_sample_names = s4_strings(bundle["atlas/patient_site"][:])
    s4_sample_counts = np.asarray(bundle["atlas/patient_site_counts"][:], dtype=np.int64)
    s4_qc_xy = np.asarray(bundle["cnv_qc/umap"][:], dtype=np.float32)
    s4_qc_cluster = s4_strings(bundle["cnv_qc/cnv_leiden"][:])
    s4_qc_celltype = s4_strings(bundle["cnv_qc/celltype"][:])
    s4_provisional_xy = np.asarray(bundle["cnv_provisional/umap"][:], dtype=np.float32)
    s4_provisional_status = s4_strings(bundle["cnv_provisional/status"][:])
    s4_marker_celltypes = s4_strings(bundle["marker_dotplots/celltypes"][:])
    s4_marker_genes = s4_strings(bundle["marker_dotplots/genes"][:])
    s4_marker_owner = s4_strings(bundle["marker_dotplots/gene_owner"][:])
    s4_marker_mean = np.asarray(bundle["marker_dotplots/mean_log1p"][:], dtype=np.float32)
    s4_marker_fraction = np.asarray(bundle["marker_dotplots/fraction_expressing"][:], dtype=np.float32)

assert s4_atlas_xy.shape == s4_qc_xy.shape == s4_provisional_xy.shape == (69_391, 2)
assert s4_sample_counts.shape == (15, 12)
assert s4_marker_mean.shape == s4_marker_fraction.shape == (12, 59)
assert s4_marker_celltypes.tolist() == S4_CT_ORDER

s4_sample_pct = 100 * s4_sample_counts / s4_sample_counts.sum(axis=1, keepdims=True)
s4_overall_pct = 100 * s4_composition_counts / s4_composition_counts.sum()
s4_status_counts = {status: int(np.sum(s4_provisional_status == status)) for status in S4_CNV_STATUS_ORDER}
assert sum(s4_status_counts.values()) == 69_391
assert s4_status_counts == {
    "provisional_normal_CNV_low": 27_421,
    "provisional_tumor_CNV_high": 41_970,
}

def s4_plot_adata(xy, values, categories, color_key, palette):
    obs = pd.DataFrame({color_key: pd.Categorical(values, categories=categories, ordered=True)})
    result = ad.AnnData(X=np.empty((len(obs), 0), dtype=np.float32), obs=obs)
    result.obsm["X_umap"] = xy
    result.uns[f"{color_key}_colors"] = np.asarray(palette, dtype=str)
    return result

s4_atlas_adata = s4_plot_adata(
    s4_atlas_xy, s4_atlas_celltype, S4_CT_ORDER, "celltype",
    [S4_CT_COLORS[key] for key in S4_CT_ORDER],
)
s4_qc_cluster_order = [str(index) for index in range(26)]
s4_qc_cluster_adata = s4_plot_adata(
    s4_qc_xy, s4_qc_cluster, s4_qc_cluster_order, "cnv_leiden", S4_CNV_CLUSTER_COLORS,
)
s4_qc_celltype_adata = s4_plot_adata(
    s4_qc_xy, s4_qc_celltype, S4_CT_ORDER, "celltype",
    [S4_CT_COLORS[key] for key in S4_CT_ORDER],
)
s4_provisional_adata = s4_plot_adata(
    s4_provisional_xy, s4_provisional_status, S4_CNV_STATUS_ORDER, "cnv_status",
    [S4_CNV_STATUS_COLORS[key] for key in S4_CNV_STATUS_ORDER],
)

def s4_clean_axis(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def s4_panel_title(ax, label, title):
    ax.set_title(f"{label}  {title}", loc="left", fontsize=S4_PANEL_TITLE_PT, fontweight="bold", pad=5)

def s4_save(fig, stem):
    outputs = {}
        path = S4_OUT / f"{stem}.{suffix}"
        fig.savefig(path, format=suffix, dpi=S4_DPI, bbox_inches="tight", facecolor="white")
        outputs[suffix] = str(path.relative_to(NOTEBOOK_DIR))
    display(fig)
    plt.close(fig)
    return outputs

def s4_celltype_handles():
    return [Line2D([0], [0], marker="o", linestyle="none", markersize=3.2,
                   markerfacecolor=S4_CT_COLORS[key], markeredgecolor="none",
                   label=S4_CT_LABEL[key]) for key in S4_CT_ORDER]

def draw_s4_t(ax):
    x = np.arange(len(s4_sample_names)); bottom = np.zeros(len(x))
    for index, celltype in enumerate(S4_CT_ORDER):
        values = s4_sample_pct[:, index]
        ax.bar(x, values, bottom=bottom, width=.78, color=S4_CT_COLORS[celltype], linewidth=0)
        bottom += values
    ax.set_xticks(x, [name.replace("|", " ") for name in s4_sample_names], rotation=48, ha="right")
    ax.set_ylim(0, 100); ax.set_ylabel("% cells"); s4_clean_axis(ax)
    s4_panel_title(ax, "T", "Cell composition by patient and site")

def draw_s4_r(ax):
    y = np.arange(len(S4_CT_ORDER))
    ax.barh(y, s4_overall_pct, color=[S4_CT_COLORS[key] for key in S4_CT_ORDER], height=.76)
    ax.set_yticks(y, [S4_CT_LABEL[key] for key in S4_CT_ORDER]); ax.invert_yaxis()
    ax.set_xlabel("% cells"); s4_clean_axis(ax); s4_panel_title(ax, "R", "Cell-type composition")

def draw_s4_s(ax):
    x = np.arange(len(s4_marker_genes))
    for row, celltype in enumerate(S4_CT_ORDER):
        sizes = S4_DOT_MIN_SIZE + S4_DOT_MAX_SIZE * np.clip(s4_marker_fraction[row], 0, 1)
        ax.scatter(x, np.full(len(x), row), s=sizes, color=S4_CT_COLORS[celltype], linewidths=0)
    ax.set_xticks(x, s4_marker_genes, rotation=90)
    ax.set_yticks(np.arange(len(S4_CT_ORDER)), [S4_CT_LABEL[key] for key in S4_CT_ORDER])
    ax.invert_yaxis(); ax.set_axisbelow(True); ax.grid(axis="x", color="#eeeeee", linewidth=.5)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    s4_panel_title(ax, "S", "Canonical marker expression")

def draw_s4_scanpy(ax, adata, color_key, title):
    collection_start = len(ax.collections)
    sc.pl.umap(
        adata, color=color_key, ax=ax, show=False, frameon=False,
        size=S4_UMAP_POINT_SIZE, alpha=S4_UMAP_ALPHA, legend_loc=None,
    )
    if S4_RASTERIZE_UMAPS:
        for collection in ax.collections[collection_start:]:
            collection.set_rasterized(True)
    ax.set_xlabel("UMAP1", fontsize=S4_AXIS_TITLE_PT); ax.set_ylabel("UMAP2", fontsize=S4_AXIS_TITLE_PT)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    s4_panel_title(ax, title.split(" ", 1)[0], title.split(" ", 1)[1])

def draw_s4_q(ax): draw_s4_scanpy(ax, s4_atlas_adata, "celltype", "Q Cell-type UMAP")
def draw_s4_n(ax): draw_s4_scanpy(ax, s4_qc_cluster_adata, "cnv_leiden", "N CNV clusters")
def draw_s4_p(ax): draw_s4_scanpy(ax, s4_qc_celltype_adata, "celltype", "P Cell type")
def draw_s4_m(ax): draw_s4_scanpy(ax, s4_provisional_adata, "cnv_status", "M CNV shortlist")

print("Figure S4 source:", S4_DATA)
print("T fields: atlas/patient_site + atlas/patient_site_counts")
print("R field: atlas/composition_counts")
print("S fields: marker_dotplots/genes + marker_dotplots/fraction_expressing")
print("Q fields: atlas/umap + atlas/celltype")
print("N fields: cnv_qc/umap + cnv_qc/cnv_leiden")
print("P fields: cnv_qc/umap + cnv_qc/celltype")
print("M fields: cnv_provisional/umap + cnv_provisional/status", s4_status_counts)

### T — Cell composition by patient and site

In [ ]:
# Panel T: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(10.2, 4.5))
draw_s4_t(ax)
if "T" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "T" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "T" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_t_outputs = s4_save(fig, "S4_T_patient_site_composition")

### R — Cell-type composition

In [ ]:
# Panel R: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(5.2, 4.5))
draw_s4_r(ax)
if "R" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "R" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "R" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_r_outputs = s4_save(fig, "S4_R_overall_composition")

### S — Canonical marker expression

In [ ]:
# Panel S: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(12.5, 5.2))
draw_s4_s(ax)
if "S" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "S" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "S" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_s_outputs = s4_save(fig, "S4_S_canonical_marker_expression")

### Q — Cell-type UMAP

In [ ]:
# Panel Q: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(6.0, 6.0))
draw_s4_q(ax)
if "Q" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "Q" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "Q" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_q_outputs = s4_save(fig, "S4_Q_celltype_umap")

### N — CNV clusters

In [ ]:
# Panel N: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(6.0, 6.0))
draw_s4_n(ax)
if "N" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "N" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "N" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_n_outputs = s4_save(fig, "S4_N_cnv_clusters")

### P — Cell type on CNV-QC UMAP

In [ ]:
# Panel P: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(6.0, 6.0))
draw_s4_p(ax)
if "P" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "P" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "P" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_p_outputs = s4_save(fig, "S4_P_cnv_celltype")

### M — CNV shortlist

In [ ]:
# Panel M: plotted directly from the fields documented above.
fig, ax = plt.subplots(figsize=(6.0, 6.0))
draw_s4_m(ax)
if "M" in ("Q", "P"):
    ax.legend(handles=s4_celltype_handles(), frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="Cell type")
elif "M" == "N":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3,
                      markerfacecolor=color, markeredgecolor="none", label=cluster)
               for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
    ax.legend(handles=handles, frameon=False, ncol=2, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV cluster")
elif "M" == "M":
    handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=4,
                      markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                      label=f"{S4_CNV_STATUS_LABEL[key]}: {s4_status_counts[key]:,}")
               for key in S4_CNV_STATUS_ORDER]
    ax.legend(handles=handles, frameon=False, fontsize=S4_LEGEND_PT,
              loc="upper left", bbox_to_anchor=(1.01, 1.0), title="CNV status")
s4_panel_m_outputs = s4_save(fig, "S4_M_cnv_shortlist")

### Final Figure S4 compilation

In [ ]:
# Final Figure S4 compilation — rendered again from the same in-memory data.
fig = plt.figure(figsize=(S4_WIDTH_IN, S4_HEIGHT_IN), facecolor="white")
grid = fig.add_gridspec(
    4, 12, height_ratios=[1.05, 1.45, .86, .74],
    left=.065, right=.965, bottom=.035, top=.955, hspace=.52, wspace=1.35,
)
ax_t = fig.add_subplot(grid[0, :8]); ax_r = fig.add_subplot(grid[0, 8:])
ax_s = fig.add_subplot(grid[1, :])
ax_q = fig.add_subplot(grid[2, 0:3]); ax_n = fig.add_subplot(grid[2, 3:6])
ax_p = fig.add_subplot(grid[2, 6:9]); ax_m = fig.add_subplot(grid[2, 9:12])
draw_s4_t(ax_t); draw_s4_r(ax_r); draw_s4_s(ax_s)
draw_s4_q(ax_q); draw_s4_n(ax_n); draw_s4_p(ax_p); draw_s4_m(ax_m)

legend_q = fig.add_subplot(grid[3, 0:3]); legend_n = fig.add_subplot(grid[3, 3:6])
legend_p = fig.add_subplot(grid[3, 6:9]); legend_m = fig.add_subplot(grid[3, 9:12])
for axis in (legend_q, legend_n, legend_p, legend_m): axis.axis("off")
legend_q.legend(handles=s4_celltype_handles(), title="Q cell type", frameon=False,
                fontsize=S4_LEGEND_PT, title_fontsize=S4_LEGEND_PT, loc="upper left")
cluster_handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=2.6,
                          markerfacecolor=color, markeredgecolor="none", label=cluster)
                   for cluster, color in zip(s4_qc_cluster_order, S4_CNV_CLUSTER_COLORS)]
legend_n.legend(handles=cluster_handles, title="N cluster", frameon=False, ncol=2,
                fontsize=S4_LEGEND_PT, title_fontsize=S4_LEGEND_PT, loc="upper left")
legend_p.legend(handles=s4_celltype_handles(), title="P cell type", frameon=False,
                fontsize=S4_LEGEND_PT, title_fontsize=S4_LEGEND_PT, loc="upper left")
status_handles = [Line2D([0], [0], marker="o", linestyle="none", markersize=3.2,
                         markerfacecolor=S4_CNV_STATUS_COLORS[key], markeredgecolor="none",
                         label=S4_CNV_STATUS_LABEL[key]) for key in S4_CNV_STATUS_ORDER]
legend_m.legend(handles=status_handles, title="M CNV status", frameon=False,
                fontsize=S4_LEGEND_PT, title_fontsize=S4_LEGEND_PT, loc="upper left")

fig.suptitle("Figure S4. Cell-type annotation and CNV quality control",
             fontsize=12, fontweight="bold", y=.985)
compiled_outputs = {}
    path = S4_COMPILED.with_suffix("." + suffix)
    fig.savefig(path, format=suffix, dpi=S4_DPI, facecolor="white")
    compiled_outputs[suffix] = str(path.relative_to(NOTEBOOK_DIR))
display(fig)
plt.close(fig)

panel_outputs = {
    "T": s4_panel_t_outputs, "R": s4_panel_r_outputs, "S": s4_panel_s_outputs,
    "Q": s4_panel_q_outputs, "N": s4_panel_n_outputs, "P": s4_panel_p_outputs,
    "M": s4_panel_m_outputs,
}
manifest = {
    "notebook": "Figure_4.ipynb", "mode": "plot_only",
    "analysis_recomputed": False, "source": str(S4_DATA.relative_to(NOTEBOOK_DIR)),
    "panel_order": ["T", "R", "S", "Q", "N", "P", "M"],
    "fields": {
        "T": ["atlas/patient_site", "atlas/patient_site_counts"],
        "R": ["atlas/composition_counts"],
        "S": ["marker_dotplots/genes", "marker_dotplots/fraction_expressing"],
        "Q": ["atlas/umap", "atlas/celltype"],
        "N": ["cnv_qc/umap", "cnv_qc/cnv_leiden"],
        "P": ["cnv_qc/umap", "cnv_qc/celltype"],
        "M": ["cnv_provisional/umap", "cnv_provisional/status"],
    },
    "renderers": {"T": "matplotlib", "R": "matplotlib", "S": "matplotlib",
                  "Q": "scanpy", "N": "scanpy", "P": "scanpy", "M": "scanpy"},
    "export": {"svg_text": "editable" if S4_EDITABLE_SVG_TEXT else "paths",
               "rasterized_panels": ["Q", "N", "P", "M"] if S4_RASTERIZE_UMAPS else [],
               "raster_dpi": S4_DPI},
    "individual_outputs": panel_outputs, "compiled_outputs": compiled_outputs,
    "checks": {"cells": 69391, "specimens": 15, "markers": 59, "cnv_clusters": 26,
               "cnv_low": 27421, "cnv_high": 41970},
}
S4_MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")
print("Compiled Figure S4:", compiled_outputs)
print("Manifest:", S4_MANIFEST)